In [ ]:
from gettext import install
import pip

%pip install requests beautifulsoup4 pandas openpyxl lxml selenium


In [ ]:
# Ultimate Tennis Statistics GOAT List - Complete Pagination

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

driver = webdriver.Chrome()

try:
    driver.get("https://www.ultimatetennisstatistics.com/goatList")
    
    wait = WebDriverWait(driver, 20)
    table = wait.until(EC.presence_of_element_located((By.ID, "goatListTable")))
    time.sleep(3)

    # Find and open the column selector dropdown (BTN-GROUP 6)
    print("Opening column selector dropdown...")
    try:
        header = driver.find_element(By.CLASS_NAME, "bootgrid-header")
        actionbar = header.find_element(By.CLASS_NAME, "actionBar")
        btn_groups = actionbar.find_elements(By.CLASS_NAME, "btn-group")
        
        # The sixth btn-group (index 5) is the column selector
        column_btn_group = btn_groups[5]
        
        # Find the dropdown button (has the glyphicon-th-list icon)
        dropdown_button = column_btn_group.find_element(By.CSS_SELECTOR, "button.dropdown-toggle")
        print(f"Found dropdown button: {dropdown_button.get_attribute('class')}")
        
        # Click to open the dropdown
        dropdown_button.click()
        time.sleep(1)

        # Find the dropdown menu
        dropdown_menu = column_btn_group.find_element(By.CSS_SELECTOR, "ul.dropdown-menu")
        
        # Find all checkboxes inside <label>
        labels = dropdown_menu.find_elements(By.TAG_NAME, "label")
        print(f"Found {len(labels)} column options")

        # Check all checkboxes
        checked_count = 0
        for i, label in enumerate(labels):
            try:
                checkbox = label.find_element(By.CSS_SELECTOR, "input[type='checkbox']")
                
                if not checkbox.is_selected():
                    # Click on the label (more reliable than clicking the checkbox)
                    driver.execute_script("arguments[0].click();", checkbox)
                    checked_count += 1
                    print(f"  Checked: {label.text[:50]}")
                    time.sleep(0.2)
                else:
                    print(f"  Already checked: {label.text[:50]}")
                    
            except Exception as e:
                print(f"  Error with label {i}: {e}")
        
        print(f"\n Selected {checked_count} additional columns")

        # Close the dropdown
        dropdown_button.click()
        time.sleep(3)  # Wait for the table to update

    except Exception as e:
        print(f"Error selecting columns: {e}")
        import traceback
        traceback.print_exc()

    # Wait for the table to update
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#goatListTable tbody tr")))
    time.sleep(2)
    
    # Extract headers
    headers = [th.text for th in driver.find_elements(By.CSS_SELECTOR, "#goatListTable thead th")]
    print(f"\nTotal columns now: {len(headers)}")
    print(f"Headers: {headers[:10]}...")

    all_rows_data = []
    page = 1

    while True:
        print(f"\nScraping page {page}...")
    
        rows = driver.find_elements(By.CSS_SELECTOR, "#goatListTable tbody tr")
    
        for row in rows:
            cells = row.find_elements(By.TAG_NAME, "td")
            all_rows_data.append([cell.text for cell in cells])
    
        print(f"  Added {len(rows)} rows (total: {len(all_rows_data)})")
      
        try:
            pagination = driver.find_element(By.CSS_SELECTOR, "ul.pagination")
            next_buttons = pagination.find_elements(By.XPATH, ".//a[contains(text(), '>')]")
        
            if not next_buttons:
                print("No next button found")
                break
        
            next_button = next_buttons[0]
            parent_li = next_button.find_element(By.XPATH, "..")
        
            if "disabled" in parent_li.get_attribute("class"):
                print("Next button is disabled")
                break
        
            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(2)
            page += 1
        
        except Exception as e:
            print(f"Pagination ended: {e}")
            break

    # Create DataFrame
    df = pd.DataFrame(all_rows_data, columns=headers)
    #df.to_csv('goat_list_complete.csv', index=False)
    
    print(f"Scraping completed!")
    print(f"Total players: {len(df)}")
    print(f"Total pages: {page}")
    print(f"Total columns: {len(headers)}")
    print("\nFirst 3 players:")
    print(df.head(3))
    
    
finally:
    driver.quit()